In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import label
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [6]:
import random as rng

def get_gaussian(mu,sigma):
    def gaussian(x):
        a = -1/2 * (x-mu)**2/sigma**2
        return np.exp(a)/(sigma*np.sqrt(2*np.pi))
    return np.vectorize(gaussian)
def get_ReLU(freq):
    def ReLU(x):
        return abs((x) * (x > 1-freq))
    return ReLU

class simulacra_dataset(Dataset):
    """simulacra of simulated data of neutrinos dataset."""

    def __init__(self, min_val, max_val, target_snr, length, seed, pusle_length):
        """
        Arguments:
            csv_file (string): Path to the csv file with annotations.
            root_dir (string): Directory with all the images.
            transform (callable, optional): Optional transform to be applied
                on a sample.
        """
        self.min_val = min_val
        self.max_val = max_val
        self.target_snr = target_snr
        self.length = length
        self.seed = seed
        self.pusle_length = pusle_length

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        np.random.seed(idx+self.seed)
        relu1 = get_ReLU(0.2)
        relu2 = get_ReLU(0.3)
        mu1 = random.randint(self.min_val,self.max_val)
        sigma1 = random.randint(10,30)
        mu2 = random.randint(self.min_val,self.max_val)
        sigma2 = random.randint(10,30)
        weight1 = relu1(rng.random())
        weight2 = relu2(rng.random())
        pulse = [get_gaussian(mu1,sigma1),get_gaussian(mu1+sigma1,sigma1),get_gaussian(mu2,sigma2)]

        mu, sigma = 0, 0.1 # mean and standard deviation
        s = np.random.normal(mu, sigma, self.pusle_length)
        a = np.linspace(0,self.pusle_length,self.pusle_length)
        wave = pulse[0](a) - pulse[1](a)*weight1 + pulse[2](a)*weight2
        snr = max(wave)/max(s)
        weight = target_snr/snr
        res = wave*weight + s
        return {
            "x": torch.from_numpy(res).float().unsqueeze(0),
            "y": torch.from_numpy(wave*weight).float().unsqueeze(0),
        }

min_val, max_val, target_snr, length, seed, pusle_length = 20,500,100,100,0,512
train_ds = simulacra_dataset(min_val, max_val, target_snr, length, seed, pusle_length)
length, seed = 100,1000
val_ds = simulacra_dataset(min_val, max_val, target_snr, length, seed, pusle_length)
length, seed = 100,2000
test_ds = simulacra_dataset(min_val, max_val, target_snr, length, seed, pusle_length)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

batch = next(iter(train_loader))
batch["x"].shape, batch["y"].shape

NameError: name 'random' is not defined